In [1]:
from google.colab import drive
drive.mount("/content/mount")

Mounted at /content/mount


In [3]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

df = pd.read_csv('태림_통합데이터3.csv')

month_map = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}
df['month_num'] = df['Month'].map(month_map)
df['target_date'] = pd.to_datetime(
    dict(year=df['Year'], month=df['month_num'], day=df['Day']), errors='coerce'
)
df['ID_Release_date'] = pd.to_datetime(
    df['ID_Release'].str[:8], format='%Y%m%d', errors='coerce'
)

#  현재 조건 — 같은 날(==)이나 미래인데 Invoice 있는 케이스는 미포함
exclude_mask = (
    (df['target_date'] <= df['ID_Release_date']) &
    (df['Firm/Forecast'] == 'FIRM') &
    (df['Shipped_Invoice'].notna())
)

df_clean = df[~exclude_mask].drop(columns=['month_num', 'target_date', 'ID_Release_date'])

# --- 엑셀 저장 ---
wb = Workbook()
ws = wb.active
ws.title = '태림_필터링데이터'

header_font = Font(name='Arial', bold=True, color='FFFFFF', size=10)
header_fill = PatternFill('solid', start_color='1F4E79')
header_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
thin = Side(style='thin', color='BFBFBF')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

cols = df_clean.columns.tolist()
for col_idx, col_name in enumerate(cols, 1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_align
    cell.border = border

even_fill = PatternFill('solid', start_color='EBF3FB')
odd_fill  = PatternFill('solid', start_color='FFFFFF')
data_font  = Font(name='Arial', size=10)
data_align = Alignment(vertical='center')

for row_idx, row in enumerate(df_clean.itertuples(index=False), 2):
    fill = even_fill if row_idx % 2 == 0 else odd_fill
    for col_idx, val in enumerate(row, 1):
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font  = data_font
        cell.fill  = fill
        cell.alignment = data_align
        cell.border = border

ws.row_dimensions[1].height = 30
ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

col_widths = {
    'ZF_PN':14, 'Supplier_PN':14, 'Order':12, 'ID_Release':18,
    'Month':10, 'Year':6, 'Day':5, 'Quantity':10, 'UN':5,
    'Firm/Forecast':14, 'Shipping_Inst':14, 'Shipped_Invoice':18, 'CUM_QTY':12
}
for col_idx, col_name in enumerate(cols, 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 12)

wb.save('태림_필터링데이터2.xlsx')